In [1]:
import subprocess, sys

pkgs = {'xgboost': 'xgboost>=2.0', 'lightgbm': 'lightgbm>=4.0'}
restart_needed = False
for mod, pkg in pkgs.items():
    try:
        __import__(mod)
        print(f'{mod} instalado ✅ — continuar')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'{mod} instalado ✅ — REINICIAR KERNEL antes de continuar')
        restart_needed = True
if not restart_needed:
    print('Listo — continuar con celda 0')


xgboost instalado ✅ — continuar
lightgbm instalado ✅ — continuar
Listo — continuar con celda 0


In [2]:
import os, sys, time, pickle, warnings, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
from scipy.optimize import differential_evolution
import mlflow
warnings.filterwarnings('ignore')

SAVE_DIR = '/srv/work'
IN_FEAT  = f'{SAVE_DIR}/te_features_v5.parquet'
OUT_DIR  = f'{SAVE_DIR}/models'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
HORIZON      = 2
LABEL_COLS   = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
ROUTE_KEYS   = ['source_agent', 'cloud_provider', 'region']
ID_COLS      = ROUTE_KEYS + ['window']
TARGET_COL   = 'label_ventana'
SPLIT_COL    = 'split'

UMBRALES = {
    'qwk':        {'floor': 0.65, 'target': 0.72, 'stretch': 0.80},
    'recall_alta':{'floor': 0.55, 'target': 0.75, 'stretch': 0.85},
    'macro_f1':   {'floor': 0.55, 'target': 0.65, 'stretch': 0.75},
    'bal_acc':    {'floor': 0.60, 'target': 0.70, 'stretch': 0.78},
}

try:
    mlflow.set_tracking_uri('http://localhost:6000')
    mlflow.set_experiment('latam2cloud')
    MLFLOW_OK = True; print('MLflow OK — http://localhost:6000')
except Exception:
    MLFLOW_OK = False; print('MLflow no disponible — continúa sin tracking')

print(f'XGBoost {xgb.__version__}  |  LightGBM {lgb.__version__}')
print(f'Input : {IN_FEAT}')
print(f'Output: {OUT_DIR}')


MLflow OK — http://localhost:6000
XGBoost 3.2.0  |  LightGBM 4.7.0
Input : /srv/work/te_features_v5.parquet
Output: /srv/work/models


In [3]:
import gc

print('Cargando etiquetas desde te_features_v5.parquet...')
df = pq.read_table(IN_FEAT).to_pandas()
print(f'  Shape: {df.shape}')

df['y'] = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)

val_mask = df['split'] == 'val'
y_val    = df.loc[val_mask, 'y'].values.astype(int)

del df; gc.collect()

class_counts_val = np.bincount(y_val, minlength=4)
print(f'\ny_val: {len(y_val):,}')
for k, name in LABEL_COLS.items():
    print(f'  {name:<8}: {class_counts_val[k]:>8,}  ({100*class_counts_val[k]/len(y_val):.1f}%)')
print('Celda 1 OK')


Cargando etiquetas desde te_features_v5.parquet...
  Shape: (3763917, 109)

y_val: 644,690
  Normal  :  551,191  (85.5%)
  Baja    :   58,645  (9.1%)
  Media   :   14,769  (2.3%)
  Alta    :   20,085  (3.1%)
Celda 1 OK


In [4]:
def evaluar(nombre, y_true, y_prob):
    y_pred = np.argmax(y_prob, axis=1)
    qwk    = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    mf1    = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bac    = balanced_accuracy_score(y_true, y_pred)
    cm     = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    n_alta = cm[3].sum()
    rec_a  = cm[3,3] / n_alta if n_alta > 0 else 0.0
    prec_a = cm[3,3] / cm[:,3].sum() if cm[:,3].sum() > 0 else 0.0
    mae_o  = np.mean(np.abs(y_true - y_pred))
    cand   = qwk >= UMBRALES['qwk']['floor'] and rec_a >= UMBRALES['recall_alta']['floor']
    sep = '=' * 110
    print(f'\n{sep}')
    print(f'  {nombre} — Validacion')
    print(f'  QWK={qwk:.4f}  MacroF1={mf1:.4f}  BalAcc={bac:.4f}  MAEord={mae_o:.4f}'
          f'  RecAlta={rec_a:.4f}  PrecAlta={prec_a:.4f}')
    if cand:
        print('  ✅ CANDIDATO H2  (QWK≥0.65=True  RecAlta≥0.55=True)')
    else:
        print(f'  ❌ No candidato  (QWK≥0.65={qwk>=0.65}  RecAlta≥0.55={rec_a>=0.55})')
    print(f'{sep}')
    print(f'  Confusion matrix (filas=real, cols=pred):')
    print(f'  {"":>12} {"Normal":>8} {"Baja":>8} {"Media":>8} {"Alta":>8}')
    for k in range(4):
        print(f'  {LABEL_COLS[k]:<12} ' + '  '.join(f'{cm[k,j]:>8,}' for j in range(4)))
    return dict(qwk=qwk, macro_f1=mf1, bal_acc=bac, recall_alta=rec_a,
                prec_alta=prec_a, mae_ord=mae_o, cm=cm, candidato=cand)

refs = {
    'rf_v5':         dict(qwk=0.6420, recall_alta=0.6726, macro_f1=0.6560, bal_acc=0.6720),
    'xgb_v5':        dict(qwk=0.6232, recall_alta=0.6648, macro_f1=0.6511, bal_acc=0.6985),
    'lgb_v5':        dict(qwk=0.6300, recall_alta=0.6824, macro_f1=0.6533, bal_acc=0.7025),
    'ens_rfxgb_v6':  dict(qwk=0.6413, recall_alta=0.5605, macro_f1=0.6563, bal_acc=0.6164),
    'xgb_v7':        dict(qwk=0.6227, recall_alta=0.6445, macro_f1=0.6503, bal_acc=0.6908),
    'lgb_v7':        dict(qwk=0.6299, recall_alta=0.6300, macro_f1=0.6543, bal_acc=0.6832),
    'ens_grid_v7':   dict(qwk=0.6453, recall_alta=0.6685, macro_f1=0.6605, bal_acc=0.6779),
}
best_prev = max(refs.items(), key=lambda x: x[1]['qwk'])
print(f'Mejor resultado previo (val):')
print(f'  {best_prev[0]}: QWK={best_prev[1]["qwk"]:.4f} | RecAlta={best_prev[1]["recall_alta"]:.4f}')
print(f'  Gap al floor H2 (QWK>=0.65): {0.65-best_prev[1]["qwk"]:.4f}')
print('Función evaluar lista.')


Mejor resultado previo (val):
  ens_grid_v7: QWK=0.6453 | RecAlta=0.6685
  Gap al floor H2 (QWK>=0.65): 0.0047
Función evaluar lista.


In [5]:
W_RF, W_XGB, W_LGB = 0.80, 0.05, 0.15

print('Cargando probabilidades guardadas en disco (v7)...')
proba_rf  = np.load(f'{OUT_DIR}/rf_v5_val_proba.npy').astype('float32')
proba_xgb = np.load(f'{OUT_DIR}/xgb_v7_val_proba.npy').astype('float32')
proba_lgb = np.load(f'{OUT_DIR}/lgb_v7_val_proba.npy').astype('float32')
print(f'  proba_rf  : {proba_rf.shape}')
print(f'  proba_xgb : {proba_xgb.shape}')
print(f'  proba_lgb : {proba_lgb.shape}')

ens_proba = (W_RF * proba_rf + W_XGB * proba_xgb + W_LGB * proba_lgb).astype('float32')
print(f'\nEnsemble base (w_RF={W_RF} w_XGB={W_XGB} w_LGB={W_LGB}) — verificación:')
res_ens_base = evaluar('Ensemble base v7 (D34) — sin threshold tuning', y_val, ens_proba)


Cargando probabilidades guardadas en disco (v7)...
  proba_rf  : (644690, 4)
  proba_xgb : (644690, 4)
  proba_lgb : (644690, 4)

Ensemble base (w_RF=0.8 w_XGB=0.05 w_LGB=0.15) — verificación:

  Ensemble base v7 (D34) — sin threshold tuning — Validacion
  QWK=0.6453  MacroF1=0.6605  BalAcc=0.6779  MAEord=0.1580  RecAlta=0.6685  PrecAlta=0.5459
  ❌ No candidato  (QWK≥0.65=False  RecAlta≥0.55=True)
  Confusion matrix (filas=real, cols=pred):
                 Normal     Baja    Media     Alta
  Normal        520,121    18,192     5,045     7,833
  Baja           15,874    39,501     1,915     1,355
  Media           4,622     1,874     6,291     1,982
  Alta            4,623       606     1,430    13,426


In [6]:

print('D35 — Threshold tuning: differential_evolution sobre bias [b_baja, b_media, b_alta]')
print(f'  Espacio: b_k ∈ [-0.5, 0.5]  |  b_normal fijo = 0')
print(f'  Baseline QWK: {res_ens_base["qwk"]:.4f}  Gap al floor: {0.65-res_ens_base["qwk"]:.4f}')
print()

_P = ens_proba.copy()
_y = y_val

def neg_qwk(b):
    b_full = np.array([0.0, b[0], b[1], b[2]], dtype='float32')
    y_pred = np.argmax(_P + b_full, axis=1)
    return -cohen_kappa_score(_y, y_pred, weights='quadratic')

t0 = time.time()
result = differential_evolution(
    neg_qwk,
    bounds=[(-0.5, 0.5)] * 3,
    seed=RANDOM_STATE,
    maxiter=300,
    popsize=20,
    tol=1e-6,
    workers=1,
    disp=False,
    mutation=(0.5, 1.0),
    recombination=0.9,
)
elapsed = time.time() - t0
b_opt    = result.x
qwk_opt  = -result.fun

print(f'Optimización: {elapsed:.1f}s  |  éxito={result.success}')
print(f'  b_baja={b_opt[0]:+.4f}  b_media={b_opt[1]:+.4f}  b_alta={b_opt[2]:+.4f}')
print(f'  QWK optimizado: {qwk_opt:.4f}  (Δ vs base: {qwk_opt-res_ens_base["qwk"]:+.4f})')

np.save(f'{OUT_DIR}/threshold_bias_v8.npy', b_opt)
print(f'\n  Bias fijado para test: {OUT_DIR}/threshold_bias_v8.npy')
print('Celda 4 OK')


D35 — Threshold tuning: differential_evolution sobre bias [b_baja, b_media, b_alta]
  Espacio: b_k ∈ [-0.5, 0.5]  |  b_normal fijo = 0
  Baseline QWK: 0.6453  Gap al floor: 0.0047

Optimización: 315.1s  |  éxito=True
  b_baja=-0.0186  b_media=+0.1140  b_alta=-0.2068
  QWK optimizado: 0.6507  (Δ vs base: +0.0054)

  Bias fijado para test: /srv/work/models/threshold_bias_v8.npy
Celda 4 OK


In [7]:
b_full = np.array([0.0, b_opt[0], b_opt[1], b_opt[2]], dtype='float32')
res_v8 = evaluar('Ensemble v8 (D34+D35 — threshold tuning)', y_val, ens_proba + b_full)

if MLFLOW_OK:
    with mlflow.start_run(run_name='ens_thresh_v8'):
        mlflow.log_params({
            'w_rf': W_RF, 'w_xgb': W_XGB, 'w_lgb': W_LGB,
            'b_baja':  round(float(b_opt[0]), 4),
            'b_media': round(float(b_opt[1]), 4),
            'b_alta':  round(float(b_opt[2]), 4),
        })
        mlflow.log_metrics({
            'qwk_val':         res_v8['qwk'],
            'recall_alta_val': res_v8['recall_alta'],
            'macro_f1_val':    res_v8['macro_f1'],
            'bal_acc_val':     res_v8['bal_acc'],
        })

modelos_v8 = {'ens_thresh_v8 (D34+D35)': res_v8}
sep = '=' * 120
print(f'\n{sep}')
print(f'  TABLA COMPARATIVA VALIDACION — v5 / v6 / v7 / v8')
print(f'{sep}')
hdr = f'  {"Modelo":<47} {"QWK":>8} {"RecAlta":>9} {"MacroF1":>9} {"BalAcc":>8}  H2?'
print(hdr); print(f'  {"-"*118}')
for nombre, r in refs.items():
    cand = r['qwk'] >= 0.65 and r['recall_alta'] >= 0.55
    print(f'  {nombre:<47} {r["qwk"]:>8.4f} {r["recall_alta"]:>9.4f}'
          f' {r["macro_f1"]:>9.4f} {r["bal_acc"]:>8.4f}  {"✅" if cand else "❌"}')
print(f'  {"─"*118}')
for nombre, r in modelos_v8.items():
    cand = r['qwk'] >= 0.65 and r['recall_alta'] >= 0.55
    print(f'  {nombre:<47} {r["qwk"]:>8.4f} {r["recall_alta"]:>9.4f}'
          f' {r["macro_f1"]:>9.4f} {r["bal_acc"]:>8.4f}  {"✅ CANDIDATO" if cand else "❌"}')
print(f'{sep}')
print(f'  Floor H2: QWK >= 0.65 AND RecAlta >= 0.55 (congelado — Kerr 1998)')
print(f'{sep}')

all_m = {**{k: dict(qwk=v['qwk'], recall_alta=v['recall_alta']) for k,v in refs.items()},
         **{k: dict(qwk=v['qwk'], recall_alta=v['recall_alta']) for k,v in modelos_v8.items()}}
candidatos = [(k, v) for k, v in all_m.items() if v['qwk'] >= 0.65 and v['recall_alta'] >= 0.55]
mejor = max(modelos_v8.items(), key=lambda x: x[1]['qwk'])

print()
if candidatos:
    best_c = max(candidatos, key=lambda x: x[1]['qwk'])
    print('✅ H2 VERIFICADA EN VAL v8')
    print(f'   Mejor candidato: {best_c[0]}')
    print(f'   QWK={best_c[1]["qwk"]:.4f}  RecAlta={best_c[1]["recall_alta"]:.4f}')
    print(f'   bias fijado: b_baja={b_opt[0]:+.4f}  b_media={b_opt[1]:+.4f}  b_alta={b_opt[2]:+.4f}')
    print()
    print('   PROTOCOLO ONE-SHOT (Kaufman et al. 2012):')
    print('   → Crear 06 evaluacion test set v8.ipynb')
    print('   → Aplicar mismo bias fijado al ensemble en test')
    print('   → Evaluar test UNA SOLA VEZ — sin ajustes posteriores')
else:
    gap = 0.65 - mejor[1]['qwk']
    print(f'❌ H2 NO VERIFICADA EN VAL v8 (0 candidatos)')
    print(f'   Mejor QWK v8 val: {mejor[1]["qwk"]:.4f}  Gap: +{gap:.4f} QWK')
    print()
    print('   Acciones siguientes:')
    print('   1. RF con n_estimators=300-400 (único backbone que llega a QWK=0.6420)')
    print('   2. Ordinal loss / QWK-proximal loss para XGB/LGB')
    print('   3. HORIZON=1 (30 min) — reduce ventaja del baseline de persistencia')

import pickle
with open(f'{OUT_DIR}/resultados_v8_val.pkl', 'wb') as f:
    pickle.dump({'res_v8': res_v8, 'b_opt': b_opt,
                 'w_rf': W_RF, 'w_xgb': W_XGB, 'w_lgb': W_LGB}, f)
print(f'\nResultados v8 guardados: {OUT_DIR}/resultados_v8_val.pkl')



  Ensemble v8 (D34+D35 — threshold tuning) — Validacion
  QWK=0.6507  MacroF1=0.6488  BalAcc=0.6670  MAEord=0.1545  RecAlta=0.4940  PrecAlta=0.6752
  ❌ No candidato  (QWK≥0.65=True  RecAlta≥0.55=False)
  Confusion matrix (filas=real, cols=pred):
                 Normal     Baja    Media     Alta
  Normal        521,071    16,772     9,868     3,480
  Baja           16,010    38,330     3,723       582
  Media           4,176     1,387     8,495       711
  Alta            5,308       499     4,357     9,921
🏃 View run ens_thresh_v8 at: http://localhost:6000/#/experiments/681062908553326840/runs/15135cc589af4b498694c1ca3cfd8837
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840

  TABLA COMPARATIVA VALIDACION — v5 / v6 / v7 / v8
  Modelo                                               QWK   RecAlta   MacroF1   BalAcc  H2?
  ----------------------------------------------------------------------------------------------------------------------
  rf_v5              